In [1]:
import pyspark, os, sys, shutil, pandas as pd
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])

PROJECT_DIR   = Path(os.getcwd())
WAREHOUSE_DIR = PROJECT_DIR / "spark-warehouse"   # Spark decides layout here  -> MANAGED tables
EXTERNAL_DIR  = PROJECT_DIR / "external-data"     # storage that WE control     -> EXTERNAL tables
LANDING_DIR   = PROJECT_DIR / "landing"           # raw files as delivered by the source system

METASTORE_DB = PROJECT_DIR / "metastore_db"

spark = (SparkSession.builder
         .appName("semantic-layering-and-meta-data-preparation")
         .config("spark.sql.warehouse.dir", WAREHOUSE_DIR.as_posix())
         .config("spark.sql.warehouse.dir", "/path/to/course/spark-warehouse")
         .config("spark.hadoop.javax.jdo.option.ConnectionURL",
                 f"jdbc:derby:;databaseName={METASTORE_DB};create=true")
         .enableHiveSupport()          # <- persistent metastore (embedded Derby in ./metastore_db)
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version          :", spark.version)
print("Catalog implementation :", spark.conf.get("spark.sql.catalogImplementation"))   # expect: hive
print("Warehouse directory    :", spark.conf.get("spark.sql.warehouse.dir"))
spark

Python /usr/local/bin/python3.12
Java /usr/lib/jvm/java-21-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/10 15:31:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version          : 4.3.0
Catalog implementation : hive
Warehouse directory    : file:/path/to/course/spark-warehouse


In [2]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

def sql(query, n=100):
    """Run Spark SQL and return the first n rows as a pandas DataFrame (pretty notebook display)."""
    return spark.sql(query).limit(n).toPandas()

def catalog_tables(db=None):
    """List tables/views registered in the catalog as a pandas DataFrame."""
    # note: listTables(db) also returns *local temp views*; we hide them when a db is given
    tables = spark.catalog.listTables(db) if db else spark.catalog.listTables()
    if db:
        tables = [t for t in tables if not t.isTemporary]
    return pd.DataFrame([t._asdict() for t in tables])

def table_summary(fqtn, keys=("Type", "Provider", "Location", "Comment", "Partition Provider",
                              "Table Properties", "Statistics", "View Text")):
    """Pick the interesting rows out of DESCRIBE TABLE EXTENDED."""
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {fqtn}").collect()
    info = {}
    for r in rows:
        name = (r.col_name or "").strip()
        if name in keys and name not in info:
            info[name] = r.data_type
    return pd.DataFrame({"property": list(info), "value": list(info.values())}).set_index("property")

def table_location(fqtn):
    """Physical location of a table, read from the metastore."""
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {fqtn}").collect()
    loc = next((r.data_type for r in rows if r.col_name == "Location"), None)
    return Path(urlparse(loc).path) if loc else None

def tree(path, max_files=3):
    """Print a directory tree (hides _SUCCESS and .crc housekeeping files). max_files=0 -> folders only."""
    path = Path(path)
    if not path.exists():
        print(f"{path}   <-- DOES NOT EXIST")
        return
    for root, dirs, files in os.walk(path):
        dirs.sort()
        depth = len(Path(root).relative_to(path).parts)
        indent = "    " * depth
        print(f"{indent}{Path(root).name}/")
        files = sorted(f for f in files if not f.startswith((".", "_")))
        for f in files[:max_files]:
            print(f"{indent}    {f}  ({(Path(root) / f).stat().st_size:,} bytes)")
        if max_files and len(files) > max_files:
            print(f"{indent}    ... {len(files) - max_files} more file(s)")

def dir_size(path):
    path = Path(path) if path else None
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) if path and path.exists() else 0

## Part 4 — Semantic layering: the Medallion architecture

```
   source export (CSV)            BRONZE                    SILVER                         GOLD  = semantic layer
  ┌──────────────────┐     ┌──────────────────┐     ┌──────────────────────┐     ┌──────────────────────────────┐
  │ landing/         │ ──► │ external table   │ ──► │ managed table        │ ──► │ dim_date, dim_product,       │
  │ online_retail/   │     │ over raw files   │     │ typed, cleaned,      │     │ dim_customer, fact_sales,    │
  │ *.csv            │     │ schema-on-read   │     │ de-duplicated,       │     │ sales_metrics_monthly,       │
  └──────────────────┘     │ source names     │     │ snake_case names,    │     │ metric_definitions           │
                           │ "as delivered"   │     │ flags + audit cols   │     │ business names & definitions │
                           └──────────────────┘     └──────────────────────┘     └──────────────────────────────┘
      who uses it:            data engineers           data engineers,               analysts, BI tools,
                              (replay, audit)          data scientists               business users
```

Each layer has a **contract** — and the contract is written down as metadata (comments, properties, grain, lineage).

## Part 4.1 — Bronze: an external table over the raw files

The source system "drops" CSV files in `landing/online_retail/`. Bronze does **not** copy or change them; it simply registers metadata on top of them:
* an **explicit schema** (no more guessing with `inferSchema`),
* **column comments** documenting what the source sends,
* **table properties**: owner, source system, refresh frequency, sensitivity.

This is **schema-on-read**: the schema is applied every time the files are read.

In [4]:
file_name = os.path.join(os.getcwd(), "data/online-retail-dataset-one-percent-sample.csv")
bronze_landing = LANDING_DIR / "online_retail"
bronze_landing.mkdir(parents=True, exist_ok=True)
shutil.copy(file_name, bronze_landing / "online_retail_2011_part_0001.csv")   # simulate the source dropping a file
tree(LANDING_DIR)

landing/
    online_retail/
        online_retail_2011_part_0001.csv  (464,095 bytes)


In [5]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze.online_retail_raw (
    InvoiceNo   STRING  COMMENT 'Invoice number as delivered. A leading C marks a cancellation',
    StockCode   STRING  COMMENT 'Product code. Non-numeric codes (POST, DOT, M, AMAZONFEE...) are not products',
    Description STRING  COMMENT 'Free-text product description, may be NULL',
    Quantity    INT     COMMENT 'Units on the line. Negative for cancellations and returns',
    InvoiceDate STRING  COMMENT 'Invoice date-time as text, format M/d/yyyy H:mm, UK local time',
    UnitPrice   DOUBLE  COMMENT 'Unit price in GBP',
    CustomerID  DOUBLE  COMMENT 'Customer identifier, exported as a float. NULL for guest checkouts',
    Country     STRING  COMMENT 'Ship-to country'
)
USING CSV
OPTIONS (header 'true', mode 'PERMISSIVE')
LOCATION '{bronze_landing.as_posix()}'
COMMENT 'Raw online-retail invoice lines exactly as exported by the e-commerce platform'
TBLPROPERTIES (
    'layer'             = 'bronze',
    'source_system'     = 'ecommerce_platform_export',
    'source_format'     = 'csv',
    'load_pattern'      = 'file drop into landing/online_retail',
    'data_owner'        = 'sales-operations',
    'data_steward'      = 'data-engineering-class',
    'contains_pii'      = 'false',
    'refresh_frequency' = 'daily'
)
""")

table_summary("bronze.online_retail_raw")

26/11/10 15:32:57 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider CSV. Persisting data source table `spark_catalog`.`bronze`.`online_retail_raw` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/11/10 15:32:57 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/11/10 15:32:57 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,value
property,
Type,EXTERNAL
Provider,CSV
Comment,Raw online-retail invoice lines exactly as exported by the e-commerce platform
Table Properties,"[contains_pii=false, data_owner=sales-operations, data_steward=data-engineering-class, layer=bronze, load_pattern=file drop into landing/online_retail, refresh_frequency=daily, source_format=csv, ..."
Location,file:/user/work/data-engineering-preparation/landing/online_retail


In [6]:
# Column-level metadata: names, types, comments
sql("DESCRIBE TABLE bronze.online_retail_raw")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,col_name,data_type,comment
0,InvoiceNo,string,Invoice number as delivered. A leading C marks a cancellation
1,StockCode,string,"Product code. Non-numeric codes (POST, DOT, M, AMAZONFEE...) are not products"
2,Description,string,"Free-text product description, may be NULL"
3,Quantity,int,Units on the line. Negative for cancellations and returns
4,InvoiceDate,string,"Invoice date-time as text, format M/d/yyyy H:mm, UK local time"
5,UnitPrice,double,Unit price in GBP
6,CustomerID,double,"Customer identifier, exported as a float. NULL for guest checkouts"
7,Country,string,Ship-to country


In [7]:
sql("SELECT * FROM bronze.online_retail_raw LIMIT 5")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,567288,22098,BOUDOIR SQUARE TISSUE BOX,4,9/19/2011 11:49,0.39,16906.0,United Kingdom
1,541827,85199S,SMALL HANGING IVORY/RED WOOD BIRD,1,1/21/2011 17:05,0.83,NaN,United Kingdom
2,574740,22027,TEA PARTY BIRTHDAY CARD,24,11/6/2011 16:07,0.42,12357.0,Switzerland
3,564182,22382,LUNCH BAG SPACEBOY DESIGN,10,8/23/2011 14:55,1.65,17242.0,United Kingdom
4,556811,21634,ASSORTED MINI MADRAS NOTEBOOK,24,6/14/2011 16:52,0.39,14997.0,United Kingdom


### 4.2 Schema-on-read and *stale* metadata
For file-based tables Spark caches the **list of files** in the session. When the source drops a new file, the catalog does not know about it until you `REFRESH TABLE`. This is another place where *metadata*, not data, determines what a query sees.

In [8]:
late_file = bronze_landing / "online_retail_2011_part_0002_late_arrival.csv"
with open(file_name) as src, open(late_file, "w") as dst:      # a tiny late batch: header + 5 lines
    for i, line in enumerate(src):
        if i > 5:
            break
        dst.write(line)

print("Count with the cached file list  :", spark.table("bronze.online_retail_raw").count(), " (probably unchanged)")
spark.sql("REFRESH TABLE bronze.online_retail_raw")
print("Count after REFRESH TABLE        :", spark.table("bronze.online_retail_raw").count())

# remove the demo file so it does not duplicate rows downstream
late_file.unlink()
spark.sql("REFRESH TABLE bronze.online_retail_raw")
print("Count after removing the file    :", spark.table("bronze.online_retail_raw").count())

Count with the cached file list  : 5419  (probably unchanged)
Count after REFRESH TABLE        : 5424
Count after removing the file    : 5419


## Part 5 — Silver: clean, typed, conformed, documented

Silver applies **technical** rules (no business opinions yet):
* consistent `snake_case` names,
* real data types: timestamp, date, `INT` customer id, `DECIMAL` for money (never float for money!),
* trimming and default values, removal of exact duplicate lines,
* **flags** that describe the data (`is_cancellation`, `is_product`) — Silver labels rows, Gold decides what to do with them,
* **audit columns** (`_source_file`, `_ingested_at`) — operational metadata stored *in* the data, which enables lineage back to the exact raw file.

In [9]:
bronze_df = spark.table("bronze.online_retail_raw")

business_columns = ["invoice_no", "stock_code", "product_description", "quantity",
                    "invoice_ts", "unit_price", "customer_id", "country"]

silver_df = (bronze_df
    .select(
        F.trim("InvoiceNo").alias("invoice_no"),
        F.upper(F.trim("StockCode")).alias("stock_code"),
        F.coalesce(F.trim("Description"), F.lit("UNKNOWN")).alias("product_description"),
        F.col("Quantity").alias("quantity"),
        F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm").alias("invoice_ts"),
        F.col("UnitPrice").cast("decimal(10,3)").alias("unit_price"),
        F.col("CustomerID").cast("int").alias("customer_id"),
        F.trim("Country").alias("country"),
        F.input_file_name().alias("_source_file"))
    .dropDuplicates(business_columns)
    .withColumn("invoice_date",    F.to_date("invoice_ts"))
    .withColumn("invoice_month",   F.date_format("invoice_ts", "yyyy-MM"))
    .withColumn("line_amount",     F.round(F.col("quantity") * F.col("unit_price"), 2).cast("decimal(12,2)"))
    .withColumn("is_cancellation", F.col("invoice_no").startswith("C"))
    .withColumn("is_product",      F.col("stock_code").rlike("^[0-9]"))
    .withColumn("_ingested_at",    F.current_timestamp())
)

print("bronze rows:", bronze_df.count(), "| silver rows after de-duplication:", silver_df.count())
silver_df.printSchema()

[Stage 13:>                                                         (0 + 1) / 1]

bronze rows: 5419 | silver rows after de-duplication: 5418
root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- product_description: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- invoice_ts: timestamp (nullable = true)
 |-- unit_price: decimal(10,3) (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- invoice_date: date (nullable = true)
 |-- invoice_month: string (nullable = true)
 |-- line_amount: decimal(12,2) (nullable = true)
 |-- is_cancellation: boolean (nullable = true)
 |-- is_product: boolean (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)



In [10]:
# A MANAGED table: the platform produces this data, so the platform owns it.
(silver_df.write
          .mode("overwrite")
          .format("parquet")
          .partitionBy("invoice_month")
          .saveAsTable("silver.sales_lines"))

tree(table_location("silver.sales_lines"), max_files=1)

sales_lines/
    invoice_month=2010-12/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (21,781 bytes)
    invoice_month=2011-01/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (19,252 bytes)
    invoice_month=2011-02/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (17,296 bytes)
    invoice_month=2011-03/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (21,164 bytes)
    invoice_month=2011-04/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (19,123 bytes)
    invoice_month=2011-05/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (21,486 bytes)
    invoice_month=2011-06/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (19,973 bytes)
    invoice_month=2011-07/
        part-00000-76cdb022-977f-45e7-8fe5-9c3d1df0ca63.c000.snappy.parquet  (22,220 bytes)
    invoice_month=2011-08/
        

### 5.1 Attach business metadata
Comments and properties are changed with `ALTER` / `COMMENT ON` statements. These are **metadata-only** operations: the Parquet files are not touched.

In [11]:
column_comments = {
    "invoice_no":          "Invoice (order) number. A leading C marks a cancellation invoice",
    "stock_code":          "Upper-cased product code",
    "product_description": "Trimmed product description; UNKNOWN when missing in the source",
    "quantity":            "Units on the line; negative for cancellations and returns",
    "invoice_ts":          "Invoice timestamp parsed from the source text (UK local time)",
    "unit_price":          "Unit price in GBP, DECIMAL(10,3)",
    "customer_id":         "Customer identifier as INT; NULL for guest checkouts",
    "country":             "Ship-to country",
    "_source_file":        "AUDIT: raw file this row was read from (lineage to bronze)",
    "invoice_date":        "Calendar date of the invoice",
    "line_amount":         "quantity x unit_price in GBP, rounded to 2 decimals",
    "is_cancellation":     "FLAG: true when invoice_no starts with C",
    "is_product":          "FLAG: true when stock_code starts with a digit (real product, not postage/fees/adjustments)",
    "_ingested_at":        "AUDIT: timestamp when this row was written to silver",
}

for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE silver.sales_lines ALTER COLUMN {col} COMMENT '{comment}'")

spark.sql("COMMENT ON TABLE silver.sales_lines IS "
          "'Cleaned online-retail invoice lines. One row per distinct invoice line. Flags describe rows, nothing is filtered out'")

spark.sql("""
ALTER TABLE silver.sales_lines SET TBLPROPERTIES (
    'layer'             = 'silver',
    'grain'             = 'one row per distinct invoice line',
    'upstream'          = 'bronze.online_retail_raw',
    'transformations'   = 'rename to snake_case; cast types; trim; default description; drop exact duplicates; add flags and audit columns',
    'data_owner'        = 'data-engineering-class',
    'contains_pii'      = 'false',
    'refresh_frequency' = 'daily'
)
""")

sql("DESCRIBE TABLE silver.sales_lines")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,col_name,data_type,comment
0,invoice_no,string,Invoice (order) number. A leading C marks a cancellation invoice
1,stock_code,string,Upper-cased product code
2,product_description,string,Trimmed product description; UNKNOWN when missing in the source
3,quantity,int,Units on the line; negative for cancellations and returns
4,invoice_ts,timestamp,Invoice timestamp parsed from the source text (UK local time)
5,unit_price,"decimal(10,3)","Unit price in GBP, DECIMAL(10,3)"
6,customer_id,int,Customer identifier as INT; NULL for guest checkouts
7,country,string,Ship-to country
8,_source_file,string,AUDIT: raw file this row was read from (lineage to bronze)
9,invoice_date,date,Calendar date of the invoice


### 5.2 Operational metadata: record data-quality results *on the table*
We compute a few data-quality counts and store them as table properties, so anyone browsing the catalog sees the health of the table without running a query.

In [12]:
dq = spark.sql("""
SELECT COUNT(*)                                             AS row_count,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
       SUM(CASE WHEN invoice_ts  IS NULL THEN 1 ELSE 0 END) AS unparsed_timestamp,
       SUM(CASE WHEN is_cancellation     THEN 1 ELSE 0 END) AS cancellation_lines,
       SUM(CASE WHEN NOT is_product      THEN 1 ELSE 0 END) AS non_product_lines,
       SUM(CASE WHEN unit_price <= 0     THEN 1 ELSE 0 END) AS non_positive_price_lines
FROM silver.sales_lines
""").first().asDict()

dq_props = ", ".join(f"'dq.{k}' = '{v}'" for k, v in dq.items())
spark.sql(f"""ALTER TABLE silver.sales_lines SET TBLPROPERTIES (
    {dq_props}, 'dq.checked_at' = '{datetime.now().isoformat(timespec="seconds")}')""")

sql("SHOW TBLPROPERTIES silver.sales_lines")

26/11/10 15:46:45 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,key,value
0,contains_pii,false
1,data_owner,data-engineering-class
2,dq.cancellation_lines,99
3,dq.checked_at,2026-11-10T15:46:45
4,dq.non_positive_price_lines,32
5,dq.non_product_lines,27
6,dq.null_customer_id,1344
7,dq.row_count,5418
8,dq.unparsed_timestamp,0
9,grain,one row per distinct invoice line


In [13]:
# The partition list is technical metadata held in the metastore
sql("SHOW PARTITIONS silver.sales_lines")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,partition
0,invoice_month=2010-12
1,invoice_month=2011-01
2,invoice_month=2011-02
3,invoice_month=2011-03
4,invoice_month=2011-04
5,invoice_month=2011-05
6,invoice_month=2011-06
7,invoice_month=2011-07
8,invoice_month=2011-08
9,invoice_month=2011-09


## Part 6 — Metadata deep dive: statistics feed the optimizer

`ANALYZE TABLE` scans the data once and stores **statistics** in the metastore: table size and row count, and per-column min / max / null count / distinct count. The cost-based optimizer uses them (e.g. to decide whether a table is small enough to broadcast in a join), and catalog tools show them to users.

In [41]:
print("Before ANALYZE:")
display(table_summary("silver.sales_lines", keys=("Statistics",)))

spark.sql("ANALYZE TABLE silver.sales_lines COMPUTE STATISTICS")
spark.sql("ANALYZE TABLE silver.sales_lines COMPUTE STATISTICS "
          "FOR COLUMNS quantity, unit_price, customer_id, country, invoice_date")

print("After ANALYZE:")
table_summary("silver.sales_lines", keys=("Statistics",))

Before ANALYZE:


,value
property,


26/11/07 14:53:42 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/11/07 14:53:42 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


After ANALYZE:


,value
property,
Statistics,"291916 bytes, 5418 rows"


In [42]:
# Column-level statistics, read straight from the metastore (no data scan)
sql("DESCRIBE EXTENDED silver.sales_lines quantity")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,info_name,info_value
0,col_name,quantity
1,data_type,int
2,comment,Units on the line; negative for cancellations and returns
3,min,-2000
4,max,670
5,num_nulls,0
6,distinct_count,97
7,avg_col_len,4
8,max_col_len,4
9,histogram,NULL


In [43]:
# The optimizer's view: estimated size and row count come from the metastore statistics
spark.conf.set("spark.sql.cbo.enabled", "true")      # use column stats + row counts in planning
spark.table("silver.sales_lines").filter("country = 'France'").explain(mode="cost")

== Optimized Logical Plan ==
Filter (isnotnull(country#13044) AND (country#13044 = France)), Statistics(sizeInBytes=34.1 KiB, rowCount=194)
+- Relation spark_catalog.silver.sales_lines[invoice_no#13037,stock_code#13038,product_description#13039,quantity#13040,invoice_ts#13041,unit_price#13042,customer_id#13043,country#13044,_source_file#13045,invoice_date#13046,line_amount#13047,is_cancellation#13048,is_product#13049,_ingested_at#13050,invoice_month#13051] parquet, Statistics(sizeInBytes=952.4 KiB, rowCount=5.42E+3)

== Physical Plan ==
*(1) Filter (isnotnull(country#13044) AND (country#13044 = France))
+- *(1) ColumnarToRow
   +- FileScan parquet spark_catalog.silver.sales_lines[invoice_no#13037,stock_code#13038,product_description#13039,quantity#13040,invoice_ts#13041,unit_price#13042,customer_id#13043,country#13044,_source_file#13045,invoice_date#13046,line_amount#13047,is_cancellation#13048,is_product#13049,_ingested_at#13050,invoice_month#13051] Batched: true, DataFilters: [isnotn

In [44]:
# All the metadata above, regenerated as DDL — the table's "birth certificate"
print(spark.sql("SHOW CREATE TABLE silver.sales_lines").first()[0])

CREATE TABLE spark_catalog.silver.sales_lines (
  invoice_no STRING COLLATE UTF8_BINARY COMMENT 'Invoice (order) number. A leading C marks a cancellation invoice',
  stock_code STRING COLLATE UTF8_BINARY COMMENT 'Upper-cased product code',
  product_description STRING COLLATE UTF8_BINARY COMMENT 'Trimmed product description; UNKNOWN when missing in the source',
  quantity INT COMMENT 'Units on the line; negative for cancellations and returns',
  invoice_ts TIMESTAMP COMMENT 'Invoice timestamp parsed from the source text (UK local time)',
  unit_price DECIMAL(10,3) COMMENT 'Unit price in GBP, DECIMAL(10,3)',
  customer_id INT COMMENT 'Customer identifier as INT; NULL for guest checkouts',
  country STRING COLLATE UTF8_BINARY COMMENT 'Ship-to country',
  _source_file STRING COLLATE UTF8_BINARY COMMENT 'AUDIT: raw file this row was read from (lineage to bronze)',
  invoice_date DATE COMMENT 'Calendar date of the invoice',
  line_amount DECIMAL(12,2) COMMENT 'quantity x unit_price in GBP, 

## Part 7 — Gold: the semantic layer

Silver is correct, but it still speaks the language of the *source system*: `stock_code`, `is_product`, `line_amount`, cancellations mixed with sales. Ask three analysts "what was revenue last month?" against Silver and you can easily get three different answers.

The **semantic layer** fixes that by publishing:
* **Dimensions** — the "by what" of analysis: `dim_date`, `dim_product`, `dim_customer`.
* **Facts** — the "how much", at a declared **grain**, with business rules already applied: `fact_sales`.
* **Metrics** — named, governed calculations: `gross_revenue`, `net_revenue`, `average_order_value`, … in `sales_metrics_monthly`, documented in a `metric_definitions` registry.

We build the semantic layer from **views**. A view stores **only a SQL definition in the metastore** — no data — so the business logic lives in one place, is always up to date with Silver, and every column carries a business description.

In [45]:
spark.sql("""
CREATE OR REPLACE VIEW gold.dim_date (
    date_key          COMMENT 'Calendar date. Join key to fact_sales.invoice_date',
    calendar_year     COMMENT 'Calendar year',
    calendar_quarter  COMMENT 'Calendar quarter 1-4',
    month_number      COMMENT 'Month number 1-12',
    month_name        COMMENT 'Month name, e.g. September',
    year_month        COMMENT 'Reporting month in yyyy-MM format',
    day_name          COMMENT 'Day of week, e.g. Monday',
    is_weekend        COMMENT 'True on Saturday and Sunday'
)
COMMENT 'Conformed calendar dimension: every day between the first and the last invoice'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'dimension',
               'grain' = 'one row per calendar day', 'upstream' = 'silver.sales_lines')
AS
SELECT d                        AS date_key,
       year(d)                  AS calendar_year,
       quarter(d)               AS calendar_quarter,
       month(d)                 AS month_number,
       date_format(d, 'MMMM')   AS month_name,
       date_format(d, 'yyyy-MM') AS year_month,
       date_format(d, 'EEEE')   AS day_name,
       dayofweek(d) IN (1, 7)   AS is_weekend
FROM (
    SELECT explode(sequence(min_d, max_d, INTERVAL 1 DAY)) AS d
    FROM (SELECT MIN(invoice_date) AS min_d, MAX(invoice_date) AS max_d FROM silver.sales_lines) AS bounds
) AS days
""")
sql("SELECT * FROM gold.dim_date ORDER BY date_key", 5)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:00 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,date_key,calendar_year,calendar_quarter,month_number,month_name,year_month,day_name,is_weekend
0,2010-12-01,2010,4,12,December,2010-12,Wednesday,False
1,2010-12-02,2010,4,12,December,2010-12,Thursday,False
2,2010-12-03,2010,4,12,December,2010-12,Friday,False
3,2010-12-04,2010,4,12,December,2010-12,Saturday,True
4,2010-12-05,2010,4,12,December,2010-12,Sunday,True


In [46]:
spark.sql("""
CREATE OR REPLACE VIEW gold.dim_product (
    product_code     COMMENT 'Code that identifies a sellable product',
    product_name     COMMENT 'Most recent description used for the product',
    list_price       COMMENT 'Most recent unit price charged, GBP',
    first_sold_date  COMMENT 'Date of the first invoice line for the product',
    last_sold_date   COMMENT 'Date of the most recent invoice line for the product'
)
COMMENT 'Product dimension: one row per sellable product. Postage, fees and manual adjustments are excluded'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'dimension',
               'grain' = 'one row per product_code', 'upstream' = 'silver.sales_lines',
               'business_rules' = 'is_product = true')
AS
SELECT stock_code                               AS product_code,
       max_by(product_description, invoice_ts)  AS product_name,
       max_by(unit_price, invoice_ts)           AS list_price,
       MIN(invoice_date)                        AS first_sold_date,
       MAX(invoice_date)                        AS last_sold_date
FROM silver.sales_lines
WHERE is_product
GROUP BY stock_code
""")
sql("SELECT * FROM gold.dim_product ORDER BY product_code", 5)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:01 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,product_code,product_name,list_price,first_sold_date,last_sold_date
0,10124G,ARMY CAMO BOOKCOVER TAPE,0.420,2011-11-06,2011-11-06
1,10133,COLOURING PENCILS BROWN TUBE,0.420,2011-01-12,2011-08-24
2,10134,UNKNOWN,0.000,2011-04-05,2011-04-05
3,10135,COLOURING PENCILS BROWN TUBE,1.250,2011-01-05,2011-05-22
4,11001,ASSTD DESIGN RACING CAR PEN,3.360,2010-12-01,2010-12-01


In [47]:
spark.sql("""
CREATE OR REPLACE VIEW gold.dim_customer (
    customer_key         COMMENT 'Customer identifier. Guest checkouts are not in this dimension',
    home_country         COMMENT 'Country of the most recent purchase',
    first_purchase_date  COMMENT 'Date of first purchase',
    last_purchase_date   COMMENT 'Date of most recent purchase',
    lifetime_orders      COMMENT 'Number of distinct non-cancelled invoices',
    customer_type        COMMENT 'Repeat when lifetime_orders is 2 or more, otherwise One-time'
)
COMMENT 'Customer dimension: one row per identified customer'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'dimension',
               'grain' = 'one row per customer_key', 'upstream' = 'silver.sales_lines',
               'contains_pii' = 'false')
AS
SELECT customer_id                                                        AS customer_key,
       max_by(country, invoice_ts)                                        AS home_country,
       MIN(invoice_date)                                                  AS first_purchase_date,
       MAX(invoice_date)                                                  AS last_purchase_date,
       COUNT(DISTINCT CASE WHEN NOT is_cancellation THEN invoice_no END)  AS lifetime_orders,
       CASE WHEN COUNT(DISTINCT CASE WHEN NOT is_cancellation THEN invoice_no END) >= 2
            THEN 'Repeat' ELSE 'One-time' END                             AS customer_type
FROM silver.sales_lines
WHERE customer_id IS NOT NULL AND is_product
GROUP BY customer_id
""")
sql("SELECT customer_type, COUNT(*) AS customers FROM gold.dim_customer GROUP BY customer_type")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:02 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,customer_type,customers
0,One-time,1138
1,Repeat,661


In [48]:
spark.sql("""
CREATE OR REPLACE VIEW gold.fact_sales (
    invoice_number     COMMENT 'Invoice (order) identifier',
    invoice_date       COMMENT 'Invoice date. Joins to dim_date.date_key',
    invoice_timestamp  COMMENT 'Invoice date and time',
    product_code       COMMENT 'Joins to dim_product.product_code',
    customer_key       COMMENT 'Joins to dim_customer.customer_key. NULL for guest checkouts',
    country            COMMENT 'Ship-to country',
    quantity           COMMENT 'Units sold; negative on returns',
    unit_price         COMMENT 'Unit price in GBP',
    sales_amount       COMMENT 'quantity x unit_price in GBP; negative on returns',
    is_return          COMMENT 'True when the line reverses an earlier sale (cancellation or negative quantity)',
    is_guest_checkout  COMMENT 'True when no customer id was captured'
)
COMMENT 'Sales fact at invoice-line grain with business rules applied: sellable products only, positive prices only'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'fact',
               'grain' = 'one row per invoice line', 'upstream' = 'silver.sales_lines',
               'business_rules' = 'is_product = true AND unit_price > 0')
AS
SELECT invoice_no                         AS invoice_number,
       invoice_date,
       invoice_ts                         AS invoice_timestamp,
       stock_code                         AS product_code,
       customer_id                        AS customer_key,
       country,
       quantity,
       unit_price,
       line_amount                        AS sales_amount,
       is_cancellation OR quantity < 0    AS is_return,
       customer_id IS NULL                AS is_guest_checkout
FROM silver.sales_lines
WHERE is_product AND unit_price > 0
""")
sql("SELECT * FROM gold.fact_sales", 5)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,invoice_number,invoice_date,invoice_timestamp,product_code,customer_key,country,quantity,unit_price,sales_amount,is_return,is_guest_checkout
0,C579090,2011-11-28,2011-11-28 10:48:00,84598,12427.0,Germany,-288,0.210,-60.48,True,False
1,C578363,2011-11-24,2011-11-24 10:40:00,21843,15482.0,United Kingdom,-72,9.950,-716.40,True,False
2,C578097,2011-11-22,2011-11-22 17:31:00,22835,NaN,United Kingdom,-48,4.250,-204.00,True,True
3,C574259,2011-11-03,2011-11-03 13:52:00,23084,15521.0,United Kingdom,-45,1.790,-80.55,True,False
4,C574057,2011-11-02,2011-11-02 14:10:00,22938,14102.0,United Kingdom,-12,1.950,-23.40,True,False


### 7.1 Governed metrics
This view is the **single source of truth** for sales KPIs. Pay attention to the *additivity* of each metric — it is part of its metadata:
* **Additive** (`gross_revenue`, `net_revenue`, `units_sold`, `orders`): safe to `SUM` across months and countries.
* **Non-additive** (`average_order_value`, `return_rate`): ratios — never sum or average them; recompute from their additive parts.
* **Distinct counts** (`active_customers`): additive across countries only if a customer has one country, and **not** additive across months (the same customer can buy in several months).

In [49]:
spark.sql("""
CREATE OR REPLACE VIEW gold.sales_metrics_monthly (
    year_month           COMMENT 'Reporting month, yyyy-MM',
    country              COMMENT 'Ship-to country',
    orders               COMMENT 'METRIC orders: distinct non-return invoices. Additive',
    active_customers     COMMENT 'METRIC active_customers: distinct identified customers with a purchase. NOT additive across months',
    units_sold           COMMENT 'METRIC units_sold: units on non-return lines. Additive',
    gross_revenue        COMMENT 'METRIC gross_revenue: sales_amount of non-return lines, GBP. Additive',
    returns_amount       COMMENT 'METRIC returns_amount: value of returned goods as a positive number, GBP. Additive',
    net_revenue          COMMENT 'METRIC net_revenue: gross_revenue minus returns_amount, GBP. Additive',
    average_order_value  COMMENT 'METRIC average_order_value: gross_revenue / orders. NON-ADDITIVE',
    return_rate          COMMENT 'METRIC return_rate: returns_amount / gross_revenue. NON-ADDITIVE'
)
COMMENT 'Governed sales KPIs at month x country grain. The single source of truth for revenue reporting'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'metrics',
               'grain' = 'one row per year_month and country',
               'upstream' = 'gold.fact_sales, gold.dim_date',
               'metric_registry' = 'gold.metric_definitions')
AS
SELECT d.year_month,
       f.country,
       COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.invoice_number END)  AS orders,
       COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.customer_key END)    AS active_customers,
       SUM(CASE WHEN NOT f.is_return THEN f.quantity ELSE 0 END)            AS units_sold,
       SUM(CASE WHEN NOT f.is_return THEN f.sales_amount ELSE 0 END)        AS gross_revenue,
       -SUM(CASE WHEN f.is_return THEN f.sales_amount ELSE 0 END)           AS returns_amount,
       SUM(f.sales_amount)                                                  AS net_revenue,
       ROUND(SUM(CASE WHEN NOT f.is_return THEN f.sales_amount ELSE 0 END)
             / NULLIF(COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.invoice_number END), 0), 2)
                                                                            AS average_order_value,
       ROUND(-SUM(CASE WHEN f.is_return THEN f.sales_amount ELSE 0 END)
             / NULLIF(SUM(CASE WHEN NOT f.is_return THEN f.sales_amount ELSE 0 END), 0), 4)
                                                                            AS return_rate
FROM gold.fact_sales AS f
JOIN gold.dim_date   AS d ON f.invoice_date = d.date_key
GROUP BY d.year_month, f.country
""")
sql("SELECT * FROM gold.sales_metrics_monthly ORDER BY year_month, net_revenue DESC", 10)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:05 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/11/07 14:54:05 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,year_month,country,orders,active_customers,units_sold,gross_revenue,returns_amount,net_revenue,average_order_value,return_rate
0,2010-12,United Kingdom,235,171,3017,7868.35,67.40,7800.95,33.48,0.0086
1,2010-12,Germany,4,4,102,112.74,0.00,112.74,28.19,0.0000
2,2010-12,Netherlands,1,1,240,100.80,0.00,100.80,100.80,0.0000
3,2010-12,France,3,3,40,85.80,0.00,85.80,28.60,0.0000
4,2010-12,Norway,1,1,60,70.20,0.00,70.20,70.20,0.0000
5,2010-12,Belgium,1,1,120,50.40,0.00,50.40,50.40,0.0000
6,2010-12,Finland,1,1,72,39.60,0.00,39.60,39.60,0.0000
7,2010-12,EIRE,2,1,18,28.50,0.00,28.50,14.25,0.0000
8,2010-12,Cyprus,1,1,12,18.40,0.00,18.40,18.40,0.0000
9,2010-12,Australia,1,1,3,17.85,0.00,17.85,17.85,0.0000


### 7.2 A metric registry — business metadata as a table
Column comments are good, but a semantic layer usually also keeps a **registry of metric definitions** that BI tools, documentation sites and people can read. This one is a small managed table.

In [50]:
metric_rows = [
    ("orders", "Orders", "Number of distinct invoices that are not returns",
     "COUNT(DISTINCT invoice_number) FILTER (WHERE NOT is_return)", "additive", "count"),
    ("active_customers", "Active customers", "Distinct identified customers with at least one purchase in the period",
     "COUNT(DISTINCT customer_key) FILTER (WHERE NOT is_return)", "distinct count - not additive across periods", "count"),
    ("units_sold", "Units sold", "Units on non-return lines",
     "SUM(quantity) FILTER (WHERE NOT is_return)", "additive", "units"),
    ("gross_revenue", "Gross revenue", "Value of goods sold before returns. Excludes postage, fees and adjustments",
     "SUM(sales_amount) FILTER (WHERE NOT is_return)", "additive", "GBP"),
    ("returns_amount", "Returns", "Value of goods returned or cancelled, as a positive number",
     "-SUM(sales_amount) FILTER (WHERE is_return)", "additive", "GBP"),
    ("net_revenue", "Net revenue", "Gross revenue minus returns",
     "SUM(sales_amount)", "additive", "GBP"),
    ("average_order_value", "Average order value", "Average gross value of an order",
     "gross_revenue / orders", "non-additive (ratio)", "GBP"),
    ("return_rate", "Return rate", "Share of gross revenue that was returned",
     "returns_amount / gross_revenue", "non-additive (ratio)", "ratio"),
]
metric_schema = "metric_name STRING, display_name STRING, business_definition STRING, " \
                "formula_sql STRING, additivity STRING, unit STRING"

(spark.createDataFrame(metric_rows, metric_schema)
      .withColumn("source_object", F.lit("gold.fact_sales"))
      .withColumn("metric_owner",  F.lit("finance-analytics"))
      .withColumn("certified",     F.lit(True))
      .write.mode("overwrite").format("parquet")
      .saveAsTable("gold.metric_definitions"))

spark.sql("COMMENT ON TABLE gold.metric_definitions IS 'Registry of governed metric definitions used by the semantic layer'")
sql("SELECT * FROM gold.metric_definitions")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,metric_name,display_name,business_definition,formula_sql,additivity,unit,source_object,metric_owner,certified
0,active_customers,Active customers,Distinct identified customers with at least one purchase in the period,COUNT(DISTINCT customer_key) FILTER (WHERE NOT is_return),distinct count - not additive across periods,count,gold.fact_sales,finance-analytics,True
1,gross_revenue,Gross revenue,"Value of goods sold before returns. Excludes postage, fees and adjustments",SUM(sales_amount) FILTER (WHERE NOT is_return),additive,GBP,gold.fact_sales,finance-analytics,True
2,returns_amount,Returns,"Value of goods returned or cancelled, as a positive number",-SUM(sales_amount) FILTER (WHERE is_return),additive,GBP,gold.fact_sales,finance-analytics,True
3,orders,Orders,Number of distinct invoices that are not returns,COUNT(DISTINCT invoice_number) FILTER (WHERE NOT is_return),additive,count,gold.fact_sales,finance-analytics,True
4,return_rate,Return rate,Share of gross revenue that was returned,returns_amount / gross_revenue,non-additive (ratio),ratio,gold.fact_sales,finance-analytics,True
5,average_order_value,Average order value,Average gross value of an order,gross_revenue / orders,non-additive (ratio),GBP,gold.fact_sales,finance-analytics,True
6,units_sold,Units sold,Units on non-return lines,SUM(quantity) FILTER (WHERE NOT is_return),additive,units,gold.fact_sales,finance-analytics,True
7,net_revenue,Net revenue,Gross revenue minus returns,SUM(sales_amount),additive,GBP,gold.fact_sales,finance-analytics,True


### 7.3 Using the semantic layer
Business questions become short, readable queries in business vocabulary — no need to know about cancellation prefixes or postage stock codes.

In [51]:
# Q1: Which countries generate the most net revenue (full period)?
sql("""
SELECT country,
       SUM(net_revenue)                                AS net_revenue,
       SUM(orders)                                     AS orders,
       ROUND(SUM(gross_revenue) / SUM(orders), 2)      AS average_order_value
FROM gold.sales_metrics_monthly
GROUP BY country
ORDER BY net_revenue DESC
""", 10)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:10 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/11/07 14:54:10 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,country,net_revenue,orders,average_order_value
0,United Kingdom,79473.08,3396,25.34
1,Netherlands,2379.75,18,137.41
2,EIRE,2238.73,61,36.87
3,Germany,1821.22,77,25.73
4,France,1766.18,72,24.77
5,Australia,482.08,6,80.62
6,Spain,367.86,19,19.36
7,Belgium,360.14,18,20.10
8,Switzerland,274.38,11,25.32
9,Portugal,264.73,10,26.47


In [52]:
# Q2: Monthly trend of gross revenue, returns and net revenue
sql("""
SELECT year_month,
       SUM(gross_revenue)  AS gross_revenue,
       SUM(returns_amount) AS returns_amount,
       SUM(net_revenue)    AS net_revenue
FROM gold.sales_metrics_monthly
GROUP BY year_month
ORDER BY year_month
""")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:10 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/11/07 14:54:10 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,year_month,gross_revenue,returns_amount,net_revenue
0,2010-12,8430.29,67.40,8362.89
1,2011-01,4998.24,14.99,4983.25
2,2011-02,5399.28,53.26,5346.02
3,2011-03,6902.94,104.08,6798.86
4,2011-04,9861.74,3776.35,6085.39
5,2011-05,8048.98,83.44,7965.54
6,2011-06,5846.54,34.50,5812.04
7,2011-07,6723.34,78.06,6645.28
8,2011-08,6057.49,103.65,5953.84
9,2011-09,8679.97,135.64,8544.33


### 7.4 Why bother? "Naive" vs. governed numbers
An analyst who goes straight to the raw table and writes the obvious query gets a different "revenue" and a different "number of orders".

In [53]:
sql("""
SELECT 'Naive: SUM(Quantity*UnitPrice) on bronze' AS approach,
       ROUND(SUM(Quantity * UnitPrice), 2)       AS revenue,
       COUNT(DISTINCT InvoiceNo)                 AS orders
FROM bronze.online_retail_raw
UNION ALL
SELECT 'Governed: gross_revenue (gold)', CAST(SUM(gross_revenue) AS DOUBLE), SUM(orders)
FROM gold.sales_metrics_monthly
UNION ALL
SELECT 'Governed: net_revenue (gold)',   CAST(SUM(net_revenue) AS DOUBLE),   SUM(orders)
FROM gold.sales_metrics_monthly
""")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:13 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/11/07 14:54:13 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,approach,revenue,orders
0,Naive: SUM(Quantity*UnitPrice) on bronze,110161.12,3886
1,Governed: gross_revenue (gold),97952.73,3744
2,Governed: net_revenue (gold),91051.26,3744


In [54]:
# Where does the difference come from? The business rules, made explicit
sql("""
SELECT CASE WHEN NOT is_product                    THEN '1. non-product lines (postage, fees, adjustments)'
            WHEN unit_price <= 0                   THEN '2. zero-price lines'
            WHEN is_cancellation OR quantity < 0   THEN '3. returns / cancellations'
            ELSE                                        '4. regular product sales'
       END                AS bucket,
       COUNT(*)           AS lines,
       SUM(line_amount)   AS amount_gbp
FROM silver.sales_lines
GROUP BY 1
ORDER BY 1
""")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:14 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,bucket,lines,amount_gbp
0,"1. non-product lines (postage, fees, adjustments)",27,19107.76
1,2. zero-price lines,31,0.00
2,3. returns / cancellations,92,-6901.47
3,4. regular product sales,5268,97952.73


### 7.5 The non-additive trap
`average_order_value` is stored per month × country. Averaging those averages weights a country with 1 order the same as the UK with hundreds. The metadata ("NON-ADDITIVE") tells the analyst what to do instead.

In [55]:
sql("""
SELECT 'WRONG: AVG(average_order_value)'          AS method, ROUND(AVG(average_order_value), 2)          AS aov
FROM gold.sales_metrics_monthly
UNION ALL
SELECT 'RIGHT: SUM(gross_revenue) / SUM(orders)', ROUND(SUM(gross_revenue) / SUM(orders), 2)
FROM gold.sales_metrics_monthly
""")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/07 14:54:16 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/11/07 14:54:16 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,method,aov
0,WRONG: AVG(average_order_value),36.83
1,RIGHT: SUM(gross_revenue) / SUM(orders),26.16


### 7.6 Views are pure metadata
A view has `Type: VIEW`, no `Location`, and its SQL text stored in the metastore. Look at `gold.db` in the warehouse: only the **managed table** `metric_definitions` has files — the five views have none.

In [56]:
table_summary("gold.sales_metrics_monthly", keys=("Type", "Comment", "Table Properties", "View Text"))

,value
property,
Type,VIEW
Comment,Governed sales KPIs at month x country grain. The single source of truth for revenue reporting
View Text,"SELECT d.year_month,\n f.country,\n COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.invoice_number END) AS orders,\n COUNT(DISTINCT CASE WHEN NOT f.is_return THEN f.customer_key..."
Table Properties,"[grain=one row per year_month and country, layer=gold, metric_registry=gold.metric_definitions, object_type=metrics, transient_lastDdlTime=1794063245, upstream=gold.fact_sales, gold.dim_date]"


In [57]:
tree(WAREHOUSE_DIR / "gold.db", max_files=1)
catalog_tables("gold")

/app/work/data-engineering-preparation/spark-warehouse/gold.db   <-- DOES NOT EXIST


,name,catalog,namespace,description,tableType,isTemporary
0,dim_customer,spark_catalog,[gold],Customer dimension: one row per identified customer,VIEW,False
1,dim_date,spark_catalog,[gold],Conformed calendar dimension: every day between the first and the last invoice,VIEW,False
2,dim_product,spark_catalog,[gold],"Product dimension: one row per sellable product. Postage, fees and manual adjustments are excluded",VIEW,False
3,fact_sales,spark_catalog,[gold],"Sales fact at invoice-line grain with business rules applied: sellable products only, positive prices only",VIEW,False
4,metric_definitions,spark_catalog,[gold],Registry of governed metric definitions used by the semantic layer,MANAGED,False
5,sales_metrics_monthly,spark_catalog,[gold],Governed sales KPIs at month x country grain. The single source of truth for revenue reporting,VIEW,False


In [58]:
# A permanent view is stored in the metastore, so it may NOT depend on a session-only temp view
try:
    spark.sql("CREATE OR REPLACE VIEW gold.bad_view AS SELECT * FROM sales")   # 'sales' is our temp view
except Exception as e:
    print("Refused ->", str(e).splitlines()[0][:200])

Refused -> [INVALID_TEMP_OBJ_REFERENCE] Cannot create the persistent object `spark_catalog`.`gold`.`bad_view` of the type VIEW because it references to the temporary object `sales` of the type VIEW. Please make 


### 7.7 Virtual vs. materialized semantic layer
Views are always fresh but recompute on every query. For dashboards we can **materialize** a snapshot as a managed table (CTAS). Trade-off: speed vs. freshness — and now the metadata must also say *when* it was refreshed.

> Vanilla Spark Parquet tables do not support `CREATE OR REPLACE TABLE` (that needs Delta/Iceberg), so we drop and re-create.

In [59]:
spark.sql("DROP TABLE IF EXISTS gold.agg_sales_monthly")
spark.sql(f"""
CREATE TABLE gold.agg_sales_monthly
USING PARQUET
COMMENT 'Materialized snapshot of gold.sales_metrics_monthly for fast dashboards'
TBLPROPERTIES ('layer' = 'gold', 'object_type' = 'aggregate',
               'upstream' = 'gold.sales_metrics_monthly',
               'refresh_frequency' = 'daily',
               'last_refreshed_at' = '{datetime.now().isoformat(timespec="seconds")}')
AS SELECT * FROM gold.sales_metrics_monthly
""")

pd.concat({t: table_summary(f"gold.{t}", keys=("Type", "Location"))["value"]
           for t in ["sales_metrics_monthly", "agg_sales_monthly"]}, axis=1)

26/11/07 14:54:22 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/11/07 14:54:22 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,sales_metrics_monthly,agg_sales_monthly
property,,
Type,VIEW,MANAGED
Location,NaN,file:/path/to/course/spark-warehouse/gold.db/agg_sales_monthly


## Part 8 — Metadata as data: generate a data dictionary

Everything we attached is queryable through the catalog API. That means documentation can be **generated** instead of hand-written — and kept in its own `governance` database. This is, in miniature, what tools like DataHub, Amundsen, OpenMetadata, Unity Catalog or Glue do.

In [60]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS governance
COMMENT 'Metadata about our data: data dictionary and catalog snapshots'
WITH DBPROPERTIES ('layer' = 'governance', 'data_owner' = 'data-engineering-class')
""")

def build_data_dictionary(databases):
    rows = []
    for db in databases:
        for t in spark.catalog.listTables(db):
            if t.isTemporary:
                continue
            fqtn  = f"{db}.{t.name}"
            props = {r.key: r.value for r in spark.sql(f"SHOW TBLPROPERTIES {fqtn}").collect()}
            for c in spark.catalog.listColumns(fqtn):
                rows.append((db, t.name, t.tableType, t.description,
                             props.get("layer"), props.get("grain"), props.get("upstream"),
                             c.name, c.dataType, c.nullable, c.isPartition, c.description))
    schema = ("database STRING, table_name STRING, table_type STRING, table_description STRING, "
              "layer STRING, grain STRING, upstream STRING, "
              "column_name STRING, data_type STRING, is_nullable BOOLEAN, is_partition BOOLEAN, "
              "column_description STRING")
    return spark.createDataFrame(rows, schema).withColumn("snapshot_at", F.current_timestamp())

data_dictionary = build_data_dictionary(["bronze", "silver", "gold"])
data_dictionary.write.mode("overwrite").format("parquet").saveAsTable("governance.data_dictionary")

sql("SELECT database, table_name, table_type, column_name, data_type, column_description "
    "FROM governance.data_dictionary ORDER BY database, table_name", 200)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,database,table_name,table_type,column_name,data_type,column_description
0,bronze,online_retail_raw,EXTERNAL,InvoiceNo,string,Invoice number as delivered. A leading C marks a cancellation
1,bronze,online_retail_raw,EXTERNAL,StockCode,string,"Product code. Non-numeric codes (POST, DOT, M, AMAZONFEE...) are not products"
2,bronze,online_retail_raw,EXTERNAL,Description,string,"Free-text product description, may be NULL"
3,bronze,online_retail_raw,EXTERNAL,Quantity,int,Units on the line. Negative for cancellations and returns
4,bronze,online_retail_raw,EXTERNAL,InvoiceDate,string,"Invoice date-time as text, format M/d/yyyy H:mm, UK local time"
5,bronze,online_retail_raw,EXTERNAL,UnitPrice,double,Unit price in GBP
6,bronze,online_retail_raw,EXTERNAL,CustomerID,double,"Customer identifier, exported as a float. NULL for guest checkouts"
7,bronze,online_retail_raw,EXTERNAL,Country,string,Ship-to country
8,gold,agg_sales_monthly,MANAGED,year_month,string,"Reporting month, yyyy-MM"
9,gold,agg_sales_monthly,MANAGED,country,string,Ship-to country


In [61]:
# Metadata quality: how well documented is each layer?
# Expect gaps on purpose: gold.metric_definitions columns and silver's partition column
# (invoice_month) were never commented. See Exercise 5.
sql("""
SELECT database,
       table_name,
       table_type,
       COUNT(*)                                                          AS columns,
       SUM(CASE WHEN column_description IS NULL OR column_description = '' THEN 1 ELSE 0 END) AS undocumented_columns
FROM governance.data_dictionary
GROUP BY database, table_name, table_type
ORDER BY database, table_name
""")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,database,table_name,table_type,columns,undocumented_columns
0,bronze,online_retail_raw,EXTERNAL,8,0
1,gold,agg_sales_monthly,MANAGED,10,0
2,gold,dim_customer,VIEW,6,0
3,gold,dim_date,VIEW,8,0
4,gold,dim_product,VIEW,5,0
5,gold,fact_sales,VIEW,11,0
6,gold,metric_definitions,MANAGED,9,9
7,gold,sales_metrics_monthly,VIEW,10,0
8,silver,sales_lines,MANAGED,15,1


In [62]:
# Lineage, straight from the 'upstream' properties we recorded
sql("""
SELECT DISTINCT upstream AS from_object, CONCAT(database, '.', table_name) AS to_object, table_type
FROM governance.data_dictionary
WHERE upstream IS NOT NULL
ORDER BY to_object
""")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,from_object,to_object,table_type
0,gold.sales_metrics_monthly,gold.agg_sales_monthly,MANAGED
1,silver.sales_lines,gold.dim_customer,VIEW
2,silver.sales_lines,gold.dim_date,VIEW
3,silver.sales_lines,gold.dim_product,VIEW
4,silver.sales_lines,gold.fact_sales,VIEW
5,"gold.fact_sales, gold.dim_date",gold.sales_metrics_monthly,VIEW
6,bronze.online_retail_raw,silver.sales_lines,MANAGED


In [63]:
# Hand it to non-Spark users (wiki, spreadsheet, catalog tool)
data_dictionary.toPandas().to_csv("data_dictionary.csv", index=False)
print("Wrote", PROJECT_DIR / "data_dictionary.csv")

Wrote /app/work/data-engineering-preparation/data_dictionary.csv


/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Part 9 — Wrap-up

### 9.1 The whole catalog at a glance

In [64]:
pd.concat([catalog_tables(db) for db in DEMO_DATABASES], ignore_index=True)[
    ["namespace", "name", "tableType", "description"]]

,namespace,name,tableType,description
0,[lab],sales_external_v2,EXTERNAL,None
1,[lab],sales_managed_v2,MANAGED,None
2,[bronze],online_retail_raw,EXTERNAL,Raw online-retail invoice lines exactly as exported by the e-commerce platform
3,[silver],sales_lines,MANAGED,"Cleaned online-retail invoice lines. One row per distinct invoice line. Flags describe rows, nothing is filtered out"
4,[gold],agg_sales_monthly,MANAGED,Materialized snapshot of gold.sales_metrics_monthly for fast dashboards
5,[gold],dim_customer,VIEW,Customer dimension: one row per identified customer
6,[gold],dim_date,VIEW,Conformed calendar dimension: every day between the first and the last invoice
7,[gold],dim_product,VIEW,"Product dimension: one row per sellable product. Postage, fees and manual adjustments are excluded"
8,[gold],fact_sales,VIEW,"Sales fact at invoice-line grain with business rules applied: sellable products only, positive prices only"
9,[gold],metric_definitions,MANAGED,Registry of governed metric definitions used by the semantic layer


In [65]:
print("Data (managed tables) in the warehouse:")
tree(WAREHOUSE_DIR, max_files=0)
print("\nData we own (external tables):")
tree(EXTERNAL_DIR, max_files=0)
print("\nRaw landing zone (bronze external table):")
tree(LANDING_DIR)

Data (managed tables) in the warehouse:
spark-warehouse/
    cars1/

Data we own (external tables):
external-data/
    lab/
        sales_external/
            invoice_month=2010-12/
            invoice_month=2011-01/
            invoice_month=2011-02/
            invoice_month=2011-03/
            invoice_month=2011-04/
            invoice_month=2011-05/
            invoice_month=2011-06/
            invoice_month=2011-07/
            invoice_month=2011-08/
            invoice_month=2011-09/
            invoice_month=2011-10/
            invoice_month=2011-11/
            invoice_month=2011-12/

Raw landing zone (bronze external table):
landing/
    online_retail/
        online_retail_2011_part_0001.csv  (464,095 bytes)


In [ ]:
spark.stop()